# NLP Lab 1: Classical NLP & Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**Course**: ITI Natural Language Processing (NLP 101)  
**Based on Lecture**: NLP-ITI-1.pdf (Classical ML for NLP)  
**Dataset**: Kaggle SMS Spam Collection (kaggle.com/datasets/uciml/sms-spam-collection-dataset)  

---

## Objectives:
1. **Text Preprocessing**: Normalize, clean text using regular expressions (`re`), and compute string distance metrics (**Levenshtein / Edit Distance**).
2. **Feature Extraction**: Build **Bag-of-Words** and **TF-IDF** representations with N-grams (`TfidfVectorizer`).
3. **Classification**: Train a classical Machine Learning model (**Logistic Regression** / **Naive Bayes**).
4. **Evaluation & Inspection**: Evaluate performance (Accuracy, Precision, Recall, F1, Confusion Matrix) and inspect top informative features.

---



In [1]:
# Setup & Environment Requirements
import numpy as np
import pandas as pd
import re
import urllib.request
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print("Setup complete! Libraries imported successfully.")



Setup complete! Libraries imported successfully.


## Step 1: Dataset Loading (Kaggle SMS Spam Dataset)

We will load the **Kaggle SMS Spam Collection Dataset**. Each sample contains a label (`ham` or `spam`) and a text message.



In [2]:
# Download dataset directly (Kaggle SMS Spam Collection mirror)
url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

# Convert labels to binary (0 = ham, 1 = spam)
df['target'] = df['label'].map({'ham': 0, 'spam': 1})

print(f"Dataset Shape: {df.shape}")
print("\nClass Distribution:")
print(df['label'].value_counts())
df.head()



Dataset Shape: (5572, 3)

Class Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text,target
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [3]:
pip install jellyfish

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 17.0 MB/s eta 0:00:00


In [ ]:
import jellyfish

---
## Step 2: Text Preprocessing & Distance Metrics

As discussed in Lecture 1, raw text contains punctuation, uppercase characters, and noise. We need to normalize text before feature extraction.

### TODO 1: Implement Text Cleaning Function
Implement `clean_text(text)` which:
1. Converts text to lower case.
2. Removes non-alphanumeric characters (keep numbers and spaces).
3. Strips whitespace.



In [ ]:
def clean_text(text):
    # Convert text to lowercase
    cleaned = text.lower()

    # Remove non-alphanumeric characters while keeping spaces
    cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)

    # Remove leading/trailing whitespace
    cleaned = cleaned.strip()
    return cleaned

# --- Test your implementation ---
sample_raw = "CONGRATS!! You've won a $1000 gift card! Call NOW at 555-1234."
sample_cleaned = clean_text(sample_raw)
print("Raw text:    ", sample_raw)
print("Cleaned text:", sample_cleaned)

# Sanity Check
assert sample_cleaned is not None, "TODO 1 is not implemented yet!"
assert sample_cleaned == "congrats youve won a 1000 gift card call now at 5551234", f"Unexpected output: {sample_cleaned}"
print("TODO 1 Passed!")


### Edit Distance (Levenshtein Distance)

In Lecture 1, we learned about edit distances for text normalization and spell-checking.
The **Levenshtein distance** counts the minimum number of single-character operations (insertions, deletions, substitutions) required to transform one string into another.

### TODO 2: Compute Levenshtein Distance
Complete `levenshtein_distance(str1, str2)` using dynamic programming or iterative distance computation.



In [ ]:
def levenshtein_distance(str1, str2):
    # TODO 2: Compute Levenshtein distance between str1 and str2
    # Hint: Create a matrix d of size (len(str1)+1) x (len(str2)+1)
    # d[i][j] = cost of converting str1[:i] to str2[:j]

    m, n = len(str1), len(str2)
    d = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        d[i][0] = i
    for j in range(n + 1):
        d[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            # Fill d[i][j] as the minimum of (deletion, insertion, substitution)
            d[i][j] = min(
                d[i-1][j] + 1,      # deletion
                d[i][j-1] + 1,      # insertion
                d[i-1][j-1] + cost  # substitution
            )

    return d[m][n]

# --- Test your implementation ---
dist1 = levenshtein_distance("intention", "execution")
dist2 = levenshtein_distance("receive", "recieve")
print(f"Distance between 'intention' & 'execution': {dist1} (Expected: 5)")
print(f"Distance between 'receive' & 'recieve':   {dist2} (Expected: 2)")

assert dist1 == 5, "TODO 2 Failed on intention -> execution!"
assert dist2 == 2, "TODO 2 Failed on receive -> recieve!"
print("TODO 2 Passed!")



In [ ]:
def levenshtein_distance(str1, str2):
    # Using the jellyfish library for Levenshtein distance
    return jellyfish.levenshtein_distance(str1, str2)

# --- Test your implementation ---
dist1 = levenshtein_distance("intention", "execution")
dist2 = levenshtein_distance("receive", "recieve")
print(f"Distance between 'intention' & 'execution': {dist1} (Expected: 5)")
print(f"Distance between 'receive' & 'recieve':   {dist2} (Expected: 2)")

assert dist1 == 5, "Failed on intention -> execution!"
assert dist2 == 2, "Failed on receive -> recieve!"
print("Levenshtein distance function updated and passed tests!")

---
## Step 3: Feature Extraction (TF-IDF & N-grams)

Clean the entire dataset and transform raw text into numerical features using **TF-IDF (Term Frequency - Inverse Document Frequency)** with **unigrams and bigrams (`ngram_range=(1,2)`)**.

### TODO 3: Extract TF-IDF Features
1. Clean all text messages in `df['text']` using `clean_text`.
2. Split dataset into train and test splits (80% train, 20% test, random_state=42).
3. Initialize `TfidfVectorizer(max_features=1000, ngram_range=(1, 2), stop_words='english')`.
4. Fit and transform training text into `X_train_tfidf`, and transform test text into `X_test_tfidf`.



In [ ]:
# Clean text column
df['cleaned_text'] = df['text'].apply(clean_text)

# Train/Test split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_text'], df['target'], test_size=0.2, random_state=42
)

# Initialize TfidfVectorizer and transform data
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    stop_words='english'
)
X_train_tfidf = vectorizer.fit_transform(X_train_raw)
X_test_tfidf = vectorizer.transform(X_test_raw)

print(f"X_train_tfidf shape: {X_train_tfidf.shape}")
print(f"X_test_tfidf shape:  {X_test_tfidf.shape}")

assert X_train_tfidf is not None and X_train_tfidf.shape[1] == 1000, "TODO 3 Failed! Feature dimensions must equal 1000."
print("TODO 3 Passed!")


---
## Step 4: Model Training (Logistic Regression / Naive Bayes)

Now train a classical machine learning classifier on the TF-IDF feature matrix.

### TODO 4: Train Logistic Regression Classifier
1. Instantiate `LogisticRegression(random_state=42)`.
2. Train (fit) the model using `X_train_tfidf` and `y_train`.



In [ ]:
# Instantiate and train Logistic Regression model
model = LogisticRegression(random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")

assert hasattr(model, "coef_"), "TODO 4 Failed! Model has not been fitted yet."
print("TODO 4 Passed!")


---
## Step 5: Model Evaluation & Feature Inspection

### TODO 5: Evaluate Model Performance
1. Generate predictions on `X_test_tfidf` using `model.predict(...)`.
2. Compute test set **Accuracy** and **F1-Score**.
3. Print the full **Classification Report** and **Confusion Matrix**.



In [ ]:
# Evaluate model predictions
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1-Score: {f1 * 100:.2f}%")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

assert accuracy > 0.90, "TODO 5 Failed! Model accuracy should be > 90%."
print("TODO 5 Passed!")


### Inspect Top Informative Spam Words
Let's inspect which vocabulary words have the highest positive weights (most indicative of **Spam**).



In [ ]:
# Feature Inspection
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]

# Get indices of top 15 highest coefficients (Spam indicators)
top_spam_idx = np.argsort(coefficients)[-15:][::-1]

print("Top 15 Words Predicting SPAM:")
for rank, idx in enumerate(top_spam_idx, 1):
    print(f"{rank:2d}. {feature_names[idx]:<15} (Weight: {coefficients[idx]:.4f})")



---
## Summary & Takeaways

In Lab 1, you learned:
1. How to clean text and compute string edit distances (Levenshtein distance).
2. How to convert text into numerical feature matrices using **TF-IDF** & N-grams.
3. How to train a classical ML classifier (**Logistic Regression**) for spam detection.
4. How to evaluate performance metrics and inspect key feature weights.

Great job completing Lab 1!

